In [3]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# 데이터 준비
text = "hello world"     # 테스트용 대체 문장 "May the force be with you."

# 문자 단위 토크나이저
tokenizer = Tokenizer(char_level=True)
tokenizer.fit_on_texts([text])

# 문자 → 정수 인코딩
sequences = []
for i in range(1, len(text)):
    seq = text[:i+1]  # 예: 'he', 'hel', 'hell', ...
    encoded = tokenizer.texts_to_sequences([seq])[0]
    sequences.append(encoded)

# 패딩
max_len = max(len(s) for s in sequences)
sequences = pad_sequences(sequences, maxlen=max_len, padding='pre')

# 입력(X)과 출력(y) 분리
X, y = sequences[:, :-1], sequences[:, -1]

# y를 one-hot 인코딩
vocab_size = len(tokenizer.word_index) + 1
y = to_categorical(y, num_classes=vocab_size)

# 모델 구성
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=10),
    SimpleRNN(32),
    Dense(vocab_size, activation='softmax')
])

# 컴파일 및 학습
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=100, verbose=0)

# 다음 글자 예측 함수
def predict_next_char(model, tokenizer, seed_text, max_len):
    encoded = tokenizer.texts_to_sequences([seed_text])[0]
    encoded = pad_sequences([encoded], maxlen=max_len-1, padding='pre')
    pred = model.predict(encoded, verbose=0)
    index = np.argmax(pred)
    for char, idx in tokenizer.word_index.items():
        if idx == index:
            return char
    return ""

# 테스트
seed = "hello w"
next_char = predict_next_char(model, tokenizer, seed, max_len)
print(f"\n입력: '{seed}' → 다음 글자 예측: '{next_char}'")


# 테스트 인터페이스
def run_interactive_prediction(model, tokenizer, max_len):
    print("\n▶ 다음 글자 예측기")
    print("아래처럼 텍스트를 입력해보세요:")
    print("예: h, he, hello wo 등")
    print("종료하려면 'exit' 입력\n")

    while True:
        seed_text = input("입력 문장: ")
        if seed_text.lower() == 'exit':
            print("종료합니다.")
            break

        if not seed_text.strip():
            print("⚠️ 공백이 아닌 문자를 입력해주세요.")
            continue

        # 토크나이저에 없는 글자 제외
        cleaned = ''.join([c for c in seed_text if c in tokenizer.word_index])
        if not cleaned:
            print("⚠️ 유효한 문자가 포함되어 있지 않습니다.")
            continue

        encoded = tokenizer.texts_to_sequences([cleaned])[0]
        encoded = pad_sequences([encoded], maxlen=max_len-1, padding='pre')
        pred = model.predict(encoded, verbose=0)
        index = np.argmax(pred)

        # 인덱스를 문자로 변환
        next_char = ""
        for char, idx in tokenizer.word_index.items():
            if idx == index:
                next_char = char
                break

        print(f"→ 예측된 다음 글자: '{next_char}'\n")

run_interactive_prediction(model, tokenizer, max_len)

#tokenizer.word_index


입력: 'hello w' → 다음 글자 예측: 'o'

▶ 다음 글자 예측기
아래처럼 텍스트를 입력해보세요:
예: h, he, hello wo 등
종료하려면 'exit' 입력

입력 문장: he
→ 예측된 다음 글자: 'l'

입력 문장: hell
→ 예측된 다음 글자: 'o'

입력 문장: exit
종료합니다.
